In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import scipy.linalg as linalg
from src.pt import *
import ot
import numpy as np
from scipy.spatial import cKDTree

%load_ext autoreload

In [ ]:
%autoreload 2

# ---- your setup ----
locs = np.array([[i, j] for i in range(20) for j in range(20)], dtype=float)
N = locs.shape[0]
weights = np.ones(N) / N
mu = EmpiricalMeasure(locs=locs, weights=weights)

X = mu.locs
x = X[:, 0]
y = X[:, 1]

# ---- define a potential phi and a stream function psi ----
# pick smooth-ish functions on the grid
phi = np.sin(0.4 * x) * np.cos(0.3 * y)
psi = np.cos(0.2 * x) * np.sin(0.5 * y)

# analytic gradients
# grad phi = (d/dx, d/dy)
grad_phi = np.stack([
    0.4 * np.cos(0.4 * x) * np.cos(0.3 * y),
    -0.3 * np.sin(0.4 * x) * np.sin(0.3 * y),
], axis=1)

# curl / divergence-free field in 2D: grad_perp psi = (dpsi/dy, -dpsi/dx)
dpsi_dx = -0.2 * np.sin(0.2 * x) * np.sin(0.5 * y)
dpsi_dy =  0.5 * np.cos(0.2 * x) * np.cos(0.5 * y)
curl_psi = np.stack([dpsi_dy, -dpsi_dx], axis=1)

# ---- mix them and project ----
alpha = 1.0   # gradient strength
beta  = 1.0   # curl strength
v_mix = alpha * grad_phi + beta * curl_psi

tan = W2EuclideanTangent(src_measure=mu, vels=v_mix)

# project (tune k, kernel, sigma as you like)
tan_proj = project_tan(
    tan,
)

v_proj = tan_proj.vels

# ---- diagnostics ----
def mse(a, b, w):
    # weighted MSE in L2(mu)
    return float(np.sum(w[:, None] * (a - b) ** 2))

mse_to_true_grad = mse(v_proj, alpha * grad_phi, weights)
mse_to_mix       = mse(v_proj, v_mix, weights)
mse_grad_removed = mse(v_mix - v_proj, beta * curl_psi, weights)

print("L2(mu) MSE(proj, true grad part)      =", mse_to_true_grad)
print("L2(mu) MSE(proj, original mix)        =", mse_to_mix)
print("L2(mu) MSE(removed, true curl part)   =", mse_grad_removed)

# optional: how much energy is kept vs removed
E_mix  = float(np.sum(weights[:, None] * v_mix**2))
E_proj = float(np.sum(weights[:, None] * v_proj**2))
print("Energy mix =", E_mix, " Energy proj =", E_proj, " frac kept =", E_proj / E_mix)

# ---- quick visual sanity check (vector norms) ----
print("mean ||grad_phi||:", float(np.mean(np.linalg.norm(grad_phi, axis=1))))
print("mean ||curl_psi||:", float(np.mean(np.linalg.norm(curl_psi, axis=1))))
print("mean ||proj||    :", float(np.mean(np.linalg.norm(v_proj, axis=1))))

In [ ]:

plt.figure(figsize=(12, 4))
plt.subplot(1, 4, 1)
plt.title("True grad part")
plt.quiver(X[:, 0], X[:, 1], grad_phi[:, 0], grad_phi[:, 1], scale=5)
plt.subplot(1, 4, 2)
plt.title("Curl part")
plt.quiver(X[:, 0], X[:, 1], curl_psi[:, 0], curl_psi[:, 1], scale=5)
plt.subplot(1, 4, 3)
plt.title("Mixed field")
plt.quiver(X[:, 0], X[:, 1], v_mix[:, 0], v_mix[:, 1], scale=5)
plt.subplot(1, 4, 4)
plt.title("Projected grad")
plt.quiver(X[:, 0], X[:, 1], v_proj[:, 0], v_proj[:, 1], scale=5)
plt.tight_layout()
plt.show()